In [ ]:
import os
import json
from typing import Annotated, TypedDict, List
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, tools_condition

# Imports for RAG Pipeline
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

load_dotenv()
calendar_db = []

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0
)


# ==========================================
# 0. LOAD MOCK JSON DATABASES
# ==========================================

def load_json_data(file_path: str, default_fallback: dict) -> dict:
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default_fallback

student_data = load_json_data("students.json", {})
placement_db = load_json_data("placement_data.json", {"companies": {}, "drives_schedule": [], "statistics": {}})
company_data = placement_db.get("companies", {})
campus_services_db = load_json_data("campus_services.json", {})


# ==========================================
# 1. RAG PIPELINE & VECTOR STORE SETUP
# ==========================================

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
PERSIST_DIRECTORY = "./chroma_db"

def initialize_vector_store(file_path: str = "campus_policies.txt"):
    """Loads institutional documents, chunks them, and stores embeddings in ChromaDB."""
    if not os.path.exists(file_path):
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(
                "CAMPUS POLICIES AND ACADEMIC REGULATIONS 2026:\n"
                "1. Attendance Threshold: Minimum 75% attendance required.\n"
                "2. Makeup Exams: Medical exceptions allowed for attendance between 65%-74%.\n"
                "3. Placement Policy: Cutoff CGPA 6.5 with 0 active backlogs.\n"
                "4. Library Fines: ₹5/day for overdue items."
            )
    
    loader = PyPDFLoader(file_path) if file_path.endswith(".pdf") else TextLoader(file_path, encoding="utf-8")
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    docs = text_splitter.split_documents(documents)
    
    return Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=PERSIST_DIRECTORY
    )

if not os.path.exists(PERSIST_DIRECTORY):
    vector_db = initialize_vector_store("campus_policies.txt")
else:
    vector_db = Chroma(
        persist_directory=PERSIST_DIRECTORY,
        embedding_function=embeddings
    )


# ==========================================
# 2. TOOL DEFINITIONS
# ==========================================

# --- Knowledge RAG Tool ---
@tool
def query_institutional_knowledge(query: str) -> str:
    """Useful for searching official campus rules, examination regulations, placement eligibility policies, 
    library policies, and general institutional FAQs from official documents."""
    docs = vector_db.similarity_search(query, k=3)
    if not docs:
        return "No relevant information found in campus documents."
    
    context = "\n---\n".join([doc.page_content for doc in docs])
    return f"Retrieved Information from Campus Documents:\n{context}"


# --- Academic Tools ---
@tool
def get_student_academics(student_id: str) -> str:
    """Retrieves current attendance, timetables, and academic stats for a student."""
    if student_id in student_data:
        student = student_data[student_id]
        return (
            f"👤 **Student Record ({student_id})**\n"
            f"• Name: {student['name']}\n"
            f"• Branch: {student['branch']} (Year {student['year']})\n"
            f"• CGPA: {student['cgpa']}\n"
            f"• Attendance: {student['attendance_percentage']}%\n"
            f"• Active Backlogs: {student.get('backlogs', 0)}"
        )
    return f"ERROR: Student ID '{student_id}' is invalid or not found in database."


# --- Placement Tools ---
@tool
def check_placement_eligibility(student_id: str, company_name: str) -> str:
    """Checks if a student meets eligibility criteria for a specific company."""    
    if student_id in student_data and company_name in company_data:
        student = student_data[student_id]
        comp = company_data[company_name]
        
        has_backlogs = student.get("backlogs", 0) > 0
        cgpa_ok = student["cgpa"] >= comp["min_cgpa"]
        year_ok = student["year"] >= comp["min_year"]
        branch_ok = student["branch"] in comp["branch_allowed"]

        if cgpa_ok and year_ok and branch_ok and not has_backlogs:
            return f"✅ Student {student_id} ({student['name']}) is ELIGIBLE for {company_name} ({comp['role']})."
        
        reasons = []
        if not cgpa_ok: reasons.append(f"CGPA {student['cgpa']} < Required {comp['min_cgpa']}")
        if not year_ok: reasons.append(f"Year {student['year']} < Required Year {comp['min_year']}")
        if not branch_ok: reasons.append(f"Branch '{student['branch']}' not in allowed list")
        if has_backlogs: reasons.append("Active backlogs present")
        
        return f"❌ Student {student_id} is NOT eligible for {company_name}. Reasons: " + ", ".join(reasons)

    if student_id not in student_data:
        return f"Student ID '{student_id}' not found."
    return f"Company '{company_name}' not found in active placement drives."

@tool
def get_eligible_companies(student_id: str) -> str:
    """Finds all companies a student is eligible to apply for based on CGPA, year, branch, and backlogs."""
    if student_id not in student_data:
        return f"Student ID '{student_id}' not found."

    student = student_data[student_id]
    eligible = []

    for comp_name, comp in company_data.items():
        if (
            student["cgpa"] >= comp["min_cgpa"]
            and student["year"] >= comp["min_year"]
            and student["branch"] in comp["branch_allowed"]
            and student.get("backlogs", 0) == 0
        ):
            eligible.append(f"• **{comp_name}**: {comp['role']} (Min CGPA: {comp['min_cgpa']})")

    if eligible:
        return f"🎯 **Eligible Companies for {student['name']} ({student_id})**:\n" + "\n".join(eligible)
    return f"Student {student_id} is currently not eligible for active company drives."

@tool
def analyze_resume_for_job(resume_text: str, target_job_role: str) -> str:
    """Parses and evaluates candidate resumes against job descriptions, returning match percentage and missing keywords."""
    return (
        f"📄 **ATS RESUME ANALYSIS REPORT**\n"
        f"• Target Role: {target_job_role}\n"
        f"• Match Score: 82%\n"
        f"• Strengths: Strong technical foundation, relevant course projects, good CGPA.\n"
        f"• Missing Keywords: Docker, CI/CD pipelines, System Design, Microservices.\n"
        f"• Recommendation: Include a dedicated 'Projects' section highlighting LangChain or AI agent implementation."
    )

@tool
def get_company_drive_schedule(month: str = "All") -> str:
    """Retrieves upcoming placement drives, drive dates, CTC/stipend packages, and application deadlines."""
    schedules = placement_db.get("drives_schedule", [])
    if not schedules:
        return "No upcoming drive schedules recorded."
    
    out = ["📅 **UPCOMING PLACEMENT DRIVES SCHEDULE**"]
    for d in schedules:
        out.append(f"• **{d['company']}** ({d['role']}) | CTC: {d['ctc']} | Drive Date: {d['drive_date']} | Deadline: {d['deadline']}")
    return "\n".join(out)

@tool
def generate_interview_prep_guide(company_name: str, role: str) -> str:
    """Generates company-specific interview preparation roadmap, technical topics, and round formats."""
    return (
        f"🎯 **INTERVIEW PREPARATION GUIDE FOR {company_name.upper()} ({role})**\n"
        f"• Round 1: Online Assessment (Data Structures, Algorithms & Aptitude).\n"
        f"• Round 2 & 3: Technical Interviews (Trees, Graphs, OOPs, System Design basics).\n"
        f"• Round 4: HR & Cultural Fitment.\n"
        f"• Focus Areas: Practice medium-hard LeetCode problems and review core CS fundamentals."
    )


# --- Event & Calendar Tools ---
@tool
def register_for_event(student_id: str, event_name: str) -> str:
    """Registers a student for a campus event, hackathon, or workshop."""
    return f"✅ Successfully registered Student {student_id} for '{event_name}'."

@tool
def add_calendar_reminder(event_name: str, date_time: str) -> str:
    """Schedules an event and sets a reminder in the user's campus calendar."""
    return f"📅 Reminder '{event_name}' scheduled for {date_time} with a 1-hour prior alert."

@tool
def schedule_event_or_appointment(
    title: str, 
    date_time: str, 
    student_id: str = None, 
    category: str = "General"
) -> str:
    """Schedules any type of event, interview, class, exam, meeting, or reminder on the campus calendar."""
    attendee_info = f" for Student {student_id}" if student_id else ""
    calendar_db.append({
        "event": title,
        "time": date_time,
        "category": category,
        "student_id": student_id
    })
    return (
        f"✅ Event Successfully Scheduled!\n"
        f"• Title: {title}\n"
        f"• Date/Time: {date_time}\n"
        f"• Category: {category}{attendee_info}\n"
        f"• Confirmation: Calendar entry created and 1-hour prior notification set."
    )


# --- Communication Tools ---
@tool
def draft_academic_email(recipient_role: str, subject: str, reason: str, student_id: str = None) -> str:
    """Drafts formal academic correspondence to Professors, HODs, or Deans."""
    student = student_data.get(student_id, {}) if student_id else {}
    s_name = student.get("name", "Student")
    s_branch = student.get("branch", "N/A")
    
    return (
        f"📧 **DRAFTED ACADEMIC EMAIL**\n"
        f"----------------------------------------\n"
        f"To: {recipient_role}\n"
        f"Subject: {subject}\n\n"
        f"Respected Sir/Madam,\n\n"
        f"I am writing to formally request {reason}.\n"
        f"My details are provided below for your reference:\n"
        f"• Name: {s_name}\n"
        f"• Student ID: {student_id if student_id else 'N/A'}\n"
        f"• Branch: {s_branch}\n\n"
        f"I request you to kindly consider my application.\n\n"
        f"Sincerely,\n"
        f"{s_name}"
    )


# --- Student Services Tools ---
@tool
def query_student_services(service_type: str) -> str:
    """Retrieves operational details regarding campus hostel, library, transport, mess, or upcoming campus events."""
    service_key = service_type.lower().strip()
    if service_key in campus_services_db:
        data = campus_services_db[service_key]
        if isinstance(data, dict):
            details = "\n".join([f"• {k.replace('_', ' ').title()}: {v}" for k, v in data.items()])
            return f"ℹ️ **{service_key.upper()} INFORMATION**\n{details}"
        elif isinstance(data, list):
            return f"ℹ️ **{service_key.upper()} INFORMATION**\n" + json.dumps(data, indent=2)
    
    return (
        f"ℹ️ **CAMPUS HELP DESK**: Query for '{service_type}' processed. "
        f"Hostel curfew is 9:30 PM, Library is open 8:00 AM - 10:00 PM, and Morning buses arrive at 8:15 AM."
    )


# --- Domain Tool Groupings ---
placement_tools = [
    check_placement_eligibility, 
    get_eligible_companies, 
    analyze_resume_for_job, 
    get_company_drive_schedule, 
    generate_interview_prep_guide, 
    schedule_event_or_appointment
]
events_tools = [register_for_event, add_calendar_reminder, schedule_event_or_appointment]
academic_tools = [get_student_academics, schedule_event_or_appointment]
knowledge_tools = [query_institutional_knowledge]
communication_tools = [draft_academic_email]
services_tools = [query_student_services]


# ==========================================
# 3. DEFINE STATE ARCHITECTURE
# ==========================================

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    next_agent: str
    student_id: str


# ==========================================
# 4. SPECIALIZED AGENT NODES (ENHANCED)
# ==========================================

def placement_agent(state: AgentState):
    """Handles placement queries, resume analysis, drive schedules, and eligibility."""
    system_prompt = SystemMessage(
        content="""You are the Campus Placement & Career Development Agent. Your role is to guide students through company eligibility, job drives, interview prep, and ATS resume evaluations.

CORE CAPABILITIES & TOOLS:
- `check_placement_eligibility`: Verify if a student meets CGPA, year, branch, or backlog criteria for a specific company.
- `get_eligible_companies`: List all companies a student qualifies for based on their record.
- `analyze_resume_for_job`: Evaluate resume text against target roles for match score and missing keywords.
- `get_company_drive_schedule`: Provide dates, deadlines, and CTC details for upcoming placement drives.
- `generate_interview_prep_guide`: Generate company-specific interview roadmaps and round breakdowns.
- `schedule_event_or_appointment`: Schedule interviews, drive deadlines, or mock practice sessions on the calendar.

RESPONSE GUIDELINES:
1. Always present results clearly using bullet points and clear headings.
2. If a student is ineligible, explicitly state which specific criteria failed (e.g., CGPA cutoff, active backlogs, or branch mismatch).
3. Encourage actionable next steps (e.g., suggesting interview prep or scheduling calendar reminders for upcoming deadlines).
4. Do not invent company policies; rely strictly on tool outputs.
5. Do NOT execute scheduling or administrative tools unless explicitly requested by the user."""
    )
    llm_with_tools = llm.bind_tools(placement_tools)
    response = llm_with_tools.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


def events_agent(state: AgentState):
    """Handles event discovery, registration, and calendar reminders."""
    system_prompt = SystemMessage(
        content="""You are the Campus Events & Time Management Agent. Your role is to help students discover campus activities, register for workshops, and maintain an organized academic calendar.

CORE CAPABILITIES & TOOLS:
- `register_for_event`: Register a student for hackathons, workshops, or guest lectures.
- `add_calendar_reminder`: Add simple notification alerts for events or assignment deadlines.
- `schedule_event_or_appointment`: Add full events, study sessions, exams, or meetings into the user's campus calendar.

RESPONSE GUIDELINES:
1. When scheduling, confirm all details clearly: Event Title, Date/Time, Category, and Confirmation status.
2. If mandatory details (like date or time) are missing from the prompt, politely ask the user for clarification before executing the tool.
3. Keep the tone enthusiastic, supportive, and highly organized.
4. Do NOT execute scheduling or administrative tools unless explicitly requested by the user."""
    )
    llm_with_tools = llm.bind_tools(events_tools)
    response = llm_with_tools.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


def academic_agent(state: AgentState):
    """Handles academic records, timetables, and attendance tracking."""
    system_prompt = SystemMessage(
        content="""You are the Academic Progress & Records Agent. Your role is to provide transparent updates on student performance, attendance metrics, and class schedules.

CORE CAPABILITIES & TOOLS:
- `get_student_academics`: Retrieve student stats including CGPA, attendance percentage, branch, year, and active backlogs.
- `schedule_event_or_appointment`: Schedule academic consultation slots, remedial classes, or exam reminders.

RESPONSE GUIDELINES:
1. Present academic metrics using clean structured formatting (bullet points or key-value summaries).
2. Highlight attendance warnings proactively if attendance is near or below the mandatory 75% threshold.
3. Handle missing or invalid Student IDs gracefully by asking the user to provide a valid Student ID (e.g., S101, S102).
4. Do NOT execute scheduling or administrative tools unless explicitly requested by the user."""
    )
    llm_with_tools = llm.bind_tools(academic_tools)
    response = llm_with_tools.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


def knowledge_agent(state: AgentState):
    """Handles RAG queries over official handbooks and campus regulations."""
    system_prompt = SystemMessage(
        content="""You are the Official Institutional Knowledge Agent. Your role is to answer questions regarding official campus handbooks, examination rules, library regulations, and grading policies using semantic document search.

CORE CAPABILITIES & TOOLS:
- `query_institutional_knowledge`: Query the vector database for retrieved context from official campus documents.

RESPONSE GUIDELINES:
1. ALWAYS call `query_institutional_knowledge` first before formulating an answer.
2. Base your response EXCLUSIVELY on the retrieved text from campus documents. Do NOT make up rules or policies.
3. If the retrieved context does not contain the answer, explicitly state: "I couldn't find specific details regarding this in the official campus policy documents."
4. Structure policy answers clearly with section references where available.
5. Do NOT execute scheduling or administrative tools unless explicitly requested by the user."""
    )
    llm_with_tools = llm.bind_tools(knowledge_tools)
    response = llm_with_tools.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


def communication_agent(state: AgentState):
    """Handles drafting formal emails, leave applications, and academic requests."""
    system_prompt = SystemMessage(
        content="""You are the Formal Academic Communication Agent. Your role is to compose professional, well-structured email drafts and official correspondence for students contacting faculty, HODs, or Deans.

CORE CAPABILITIES & TOOLS:
- `draft_academic_email`: Generate structured email templates including Subject line, Salutation, Body text, and Formal Sign-off.

RESPONSE GUIDELINES:
1. Ensure the draft maintains a polite, professional, and academically respectful tone.
2. Clearly highlight placeholder variables (e.g., dates, reason details) if the user did not provide complete information.
3. Format output cleanly inside a blockquote or boxed structure so the student can copy and paste it easily.
4. Do NOT execute scheduling or administrative tools unless explicitly requested by the user."""
    )
    llm_with_tools = llm.bind_tools(communication_tools)
    response = llm_with_tools.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


def student_services_agent(state: AgentState):
    """Handles facility, hostel, library, transport, and campus amenity queries."""
    system_prompt = SystemMessage(
        content="""You are the Campus Student Services & Facilities Agent. Your role is to assist students with operational details regarding campus living, amenities, and daily logistics.

CORE CAPABILITIES & TOOLS:
- `query_student_services`: Look up operational information for Hostel (gate timings, mess menus), Library (hours, borrow limits), Transport (bus routes, arrival times), and General Services.

RESPONSE GUIDELINES:
1. Provide quick, scannable, and practical answers to logistics questions.
2. Include operational hours, emergency contact details, or location details whenever relevant.
3. If query covers multiple services (e.g., hostel AND transport), answer each section under a dedicated sub-heading.
4. Do NOT execute scheduling or administrative tools unless explicitly requested by the user."""
    )
    llm_with_tools = llm.bind_tools(services_tools)
    response = llm_with_tools.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


# ==========================================
# 5. ORCHESTRATOR / ROUTER AGENT (ENHANCED)
# ==========================================

def orchestrator(state: AgentState):
    """Analyzes user message and routes to the appropriate specialized agent."""

    last_msg = state["messages"][-1]
    if last_msg.type == "ai" and not getattr(last_msg, "tool_calls", None):
        return {"next_agent": "finish"}
    
    router_prompt = SystemMessage(
        content="""You are the Smart Campus System Central Orchestrator.
Your primary task is to accurately categorize incoming student queries and delegate them to exactly ONE specialized domain agent.

ROUTING TABLE & RULES:

1. 'placement_agent':
   - Queries about company eligibility, job drives, salaries/CTC, ATS resume checks, or interview preparation.
   - Example: "Am I eligible for Google?", "Review my resume for SDE", "What placement drives are scheduled this month?"

2. 'events_agent':
   - Queries related to registering for hackathons, campus events, workshops, or setting custom calendar reminders.
   - Example: "Register me for the AI Hackathon", "Remind me about project submission on Friday at 4 PM."

3. 'academic_agent':
   - Queries regarding student CGPA, attendance records, academic performance stats, or personal class schedules.
   - Example: "What is my current attendance for S101?", "Show my CGPA and backlogs."

4. 'knowledge_agent':
   - Queries about official campus policies, examination rules, makeup exam criteria, grading schemes, or library overdue fines.
   - Example: "What is the policy for medical leave during exams?", "How many books can I borrow from the library?"

5. 'communication_agent':
   - Requests to draft formal emails or leave applications to Professors, Deans, HODs, or Coordinators.
   - Example: "Draft an email to my HOD asking for leave due to illness."

6. 'student_services_agent':
   - Queries regarding hostel gate timings, mess schedules, bus/transportation routes, or campus facility locations.
   - Example: "What time does the hostel gate close?", "Show me the morning bus timetable."

7. 'FINISH':
   - Select 'FINISH' ONLY if the user's query has already been fully answered in the conversation history, or if they are saying goodbye/thank you.

CRITICAL INSTRUCTION:
Respond with EXACTLY ONE word matching one of the keys below (no punctuation, markdown, or extra words):
placement_agent
events_agent
academic_agent
knowledge_agent
communication_agent
student_services_agent
FINISH"""
    )
    
    response = llm.invoke([router_prompt] + state["messages"])
    
    if isinstance(response.content, str):
        content_text = response.content
    elif isinstance(response.content, list):
        content_text = "".join(
            part if isinstance(part, str) else part.get("text", "") if isinstance(part, dict) else getattr(part, "text", "")
            for part in response.content
        )
    else:
        content_text = str(response.content)

    route = content_text.strip().lower().replace("`", "").replace("'", "")
    valid_routes = [
        "placement_agent", "events_agent", "academic_agent", 
        "knowledge_agent", "communication_agent", "student_services_agent", "finish"
    ]
    
    if route not in valid_routes:
        route = "finish"
        
    return {"next_agent": route}

def route_decision(state: AgentState) -> str:
    """Conditional edge routing function."""
    return state["next_agent"]


# ==========================================
# 6. BUILD THE LANGGRAPH WORKFLOW
# ==========================================

workflow = StateGraph(AgentState)

# Add Agent Nodes
workflow.add_node("orchestrator", orchestrator)
workflow.add_node("placement_agent", placement_agent)
workflow.add_node("events_agent", events_agent)
workflow.add_node("academic_agent", academic_agent)
workflow.add_node("knowledge_agent", knowledge_agent)
workflow.add_node("communication_agent", communication_agent)
workflow.add_node("student_services_agent", student_services_agent)

# Add Tool Nodes
workflow.add_node("placement_tools", ToolNode(placement_tools))
workflow.add_node("events_tools", ToolNode(events_tools))
workflow.add_node("academic_tools", ToolNode(academic_tools))
workflow.add_node("knowledge_tools", ToolNode(knowledge_tools))
workflow.add_node("communication_tools", ToolNode(communication_tools))
workflow.add_node("services_tools", ToolNode(services_tools))

# Entry Edge
workflow.add_edge(START, "orchestrator")

# Conditional Routing Edges from Orchestrator
workflow.add_conditional_edges(
    "orchestrator",
    route_decision,
    {
        "placement_agent": "placement_agent",
        "events_agent": "events_agent",
        "academic_agent": "academic_agent",
        "knowledge_agent": "knowledge_agent",
        "communication_agent": "communication_agent",
        "student_services_agent": "student_services_agent",
        "finish": END
    }
)

# Agent-Tool Connections
agent_tool_pairs = [
    ("placement_agent", "placement_tools"),
    ("events_agent", "events_tools"),
    ("academic_agent", "academic_tools"),
    ("knowledge_agent", "knowledge_tools"),
    ("communication_agent", "communication_tools"),
    ("student_services_agent", "services_tools"),
]

for agent_node, tool_node in agent_tool_pairs:
    workflow.add_conditional_edges(
        agent_node,
        tools_condition,
        {
            "tools": tool_node,
            END: "orchestrator"
        }
    )
    workflow.add_edge(tool_node, agent_node)

# Checkpointer Setup
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)


# ==========================================
# 7. RUNNABLE CLI DEMO
# ==========================================

if __name__ == "__main__":
    config = {"configurable": {"thread_id": "student_session_101"}}
    
    print("🤖 --- Smart Campus Multi-Agent System --- 🤖")
    print("Type 'exit' or 'quit' to end the session.\n")

    while True:
        user_query = input("\n👤 User: ")
        if user_query.lower() in ["exit", "quit"]:
            print("Goodbye! 👋")
            break

        initial_state = {
            "messages": [HumanMessage(content=user_query)]
        }

        for event in app.stream(initial_state, config=config):
            for node_name, output in event.items():
                print(f"🔄 Executed Node: [{node_name}]")
                
                if "next_agent" in output:
                    print(f"   🔀 Routed to: {output['next_agent']}")
                
                if "messages" in output and output["messages"]:
                    last_msg = output["messages"][-1]
                    
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"   🛠️  Invoked Tool: {last_msg.tool_calls[0]['name']}")
                    elif last_msg.type == "tool":
                        print(f"   📊 Tool Result:\n{last_msg.content}")
                    elif last_msg.content:
                        print(f"   💬 Agent Output:\n{last_msg.content}")
                        
                print("-" * 50)